# Bias Mitigation Pipeline: Finetune -> Merge -> HONEST Benchmark


In [ ]:
# 1. Setup repository & environment
%cd /kaggle/working
!rm -rf Model_Merging mergekit
!git clone https://github.com/Bamboohoccode/Model_Merging.git
!git clone https://github.com/arcee-ai/mergekit.git
%cd /kaggle/working/mergekit
!pip install -e . -q
%cd /kaggle/working/Model_Merging
!pip install -r requirement.txt -q
!rm -rf ~/.cache/huggingface/hub/models--trinhkhng--*
!rm -rf /kaggle/working/*_Merged_* /kaggle/working/*_debias*


In [ ]:
# 2. Config & Secrets
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

MODEL_NAME = "openai-community/gpt2"
WORK_DIR = "/kaggle/working/"
HF_NAME = "trinhkhng"
EPOCHS = 1
LR = 3e-5

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")
login(token)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# 3. Finetune 1 Epoch, Merge (Linear & SLERP), & Run HONEST Evaluation
%cd /kaggle/working/Model_Merging

!python Finetuning_model.py \n    --name_model "{MODEL_NAME}" \n    --work_dir "{WORK_DIR}" \n    --epochs {EPOCHS} \n    --learning_rate {LR} \n    --HF_TOKEN {token}

!python merge.py \n    --name_model "{MODEL_NAME}" \n    --work_dir "{WORK_DIR}" \n    --debias_model_dir "/kaggle/working/gpt2_debias" \n    --hf_namespace "{HF_NAME}" \n    --HF_TOKEN {token} \n    --merge_methods linear slerp \n    --alphas 0.0 0.1 0.2 0.3 0.4 0.5

!python HONEST_BENCHMARK.py \n    --name_model "{MODEL_NAME}" \n    --work_dir "{WORK_DIR}" \n    --hf_namespace "{HF_NAME}" \n    --merge_methods linear slerp
